# Análisis de inversión en USDT — perspectiva del prestamista mexicano

**Pregunta nucleo:** si tienes capital disponible, ¿conviene mantenerlo en USDT (con yield) o convertirlo a MXN? ¿Cuanto valor en pesos esperar a 1, 3, 5 anios?

## Estructura

1. **Comparativa historica** — USDT vs BTC vs ETH vs MXN, retorno anualizado, volatilidad, max drawdown, Sharpe-ish, mejor/peor anio. Periodo comun y serie larga del USDMXN (2001).
2. **Monte Carlo USDMXN** — Bootstrap-KDE sobre retornos diarios para proyectar el USDMXN a 1/2/3/5 anios (5,000 trayectorias).
3. **Estrategias de yield** — Aave, Compound, lending centralizado, x2o stablecoin lending propio. Retorno MXN-equivalente bajo escenarios P5/P50/P95 del USDMXN.

## Datos
- USDMXN: Banxico (serie diaria SF63528, 2001–2026)
- BTC/ETH: Binance (1-min, resampleados a diario)


In [ ]:
import sys, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

# Reuso del motor Monte Carlo de precios_futuros
sys.path.insert(0, "../precios_futuros")
from utils.bootstrap_mc import MonteCarloPriceSimulator

CONFIG = {
    "BTC_CSV": "../../data/raw/btcusd_1-min_data.csv",
    "ETH_CSV": "../../data/raw/ethusd_1min_ohlc.csv",
    "USDMXN_XLSX": "../../data/raw/usd20012026.xlsx",

    "MC_N_SIMS": 5000,
    "MC_HORIZONS_YEARS": [1, 2, 3, 5],
    "TRADING_DAYS_PER_YEAR": 252,

    "SEED": 42,
    "CAPITAL_USD": 100_000,

    # Yields USD anuales tipicos por estrategia
    "YIELD_STRATEGIES": [
        ("Hold USDT en wallet (sin yield)", 0.00,
         "Sin riesgo de contraparte, sin retorno USD. Solo expones a FX."),
        ("Aave USDT supply", 0.06,
         "Variable; 4–8% típico segun demanda. Smart-contract risk."),
        ("Compound USDT", 0.055,
         "Similar a Aave; rates ligeramente mas bajos."),
        ("Lending centralizado (Nexo / Coinbase)", 0.085,
         "Mayor yield, riesgo de contraparte (custodia)."),
        ("x2o stablecoin lending propio (BTC/ETH-collateralized USDT loans)", 0.13,
         "Yield del modelo de negocio — mas alto, requiere operacion."),
    ],
    "YIELD_RECOMMENDED_INDEX": 1,  # Aave por liquidez/simplicidad
}

IMG_DIR = "img"
os.makedirs(IMG_DIR, exist_ok=True)
np.random.seed(CONFIG["SEED"])

print("Configuracion cargada.")


## 1. Carga de datos historicos

In [ ]:
# BTC (timestamp Unix)
btc_raw = pd.read_csv(CONFIG["BTC_CSV"])
btc_raw["date"] = pd.to_datetime(btc_raw["Timestamp"], unit="s")
btc = (btc_raw.set_index("date").sort_index()[["Close"]]
       .rename(columns={"Close": "BTC"}))

# ETH (timestamp Unix)
eth_raw = pd.read_csv(CONFIG["ETH_CSV"])
eth_raw["date"] = pd.to_datetime(eth_raw["timestamp"], unit="s")
eth = (eth_raw.set_index("date").sort_index()[["close"]]
       .rename(columns={"close": "ETH"}))

# USDMXN (Banxico — header en filas 0-17)
usdmxn_raw = pd.read_excel(CONFIG["USDMXN_XLSX"], header=None, skiprows=18,
                            names=["date", "USDMXN"])
usdmxn_raw["date"] = pd.to_datetime(usdmxn_raw["date"])
usdmxn_raw["USDMXN"] = pd.to_numeric(usdmxn_raw["USDMXN"], errors="coerce")
usdmxn_raw = usdmxn_raw.dropna(subset=["USDMXN"])
usdmxn = usdmxn_raw.set_index("date").sort_index()

# Resamplear cripto a diario
btc_daily = btc.resample("1D").last().dropna()
eth_daily = eth.resample("1D").last().dropna()

# Periodo comun
common_start = max(btc_daily.index.min(), eth_daily.index.min(), usdmxn.index.min())
common_end = min(btc_daily.index.max(), eth_daily.index.max(), usdmxn.index.max())

# Alinear (forward-fill USDMXN para fines de semana / festivos)
all_data = pd.concat([
    btc_daily.loc[common_start:common_end],
    eth_daily.loc[common_start:common_end],
    usdmxn.loc[common_start:common_end],
], axis=1).ffill().dropna()

print(f"BTC: {btc_daily.index.min().date()} a {btc_daily.index.max().date()} ({len(btc_daily):,} dias)")
print(f"ETH: {eth_daily.index.min().date()} a {eth_daily.index.max().date()} ({len(eth_daily):,} dias)")
print(f"USDMXN: {usdmxn.index.min().date()} a {usdmxn.index.max().date()} ({len(usdmxn):,} dias)")
print(f"Periodo comun: {common_start.date()} a {common_end.date()} ({len(all_data):,} dias)")


## 2. Comparativa historica de retornos

USDT vs BTC vs ETH vs MXN, en USD y en MXN. *MXN cash* es la opcion de no diversificar (mantener pesos).


In [ ]:
# Series sinteticas: USDT_USD constante; USDT_MXN = USDMXN; cripto en USD y MXN
synth = pd.DataFrame(index=all_data.index)
synth["USDT (USD)"] = 1.0
synth["MXN cash (en MXN)"] = 1.0
synth["USDT (en MXN)"] = all_data["USDMXN"]
synth["BTC (USD)"] = all_data["BTC"]
synth["ETH (USD)"] = all_data["ETH"]
synth["BTC (en MXN)"] = all_data["BTC"] * all_data["USDMXN"]
synth["ETH (en MXN)"] = all_data["ETH"] * all_data["USDMXN"]
synth["MXN cash (en USD)"] = 1.0 / all_data["USDMXN"]

TPY = CONFIG["TRADING_DAYS_PER_YEAR"]
n_years = (synth.index[-1] - synth.index[0]).days / 365.25


def asset_stats(series):
    if series.std() < 1e-12:
        return {"CAGR": 0.0, "Vol_anual": 0.0, "Max_DD": 0.0,
                "Sharpe_ish": np.nan, "Mejor_año": 0.0, "Peor_año": 0.0}
    s = series.dropna()
    cagr = (s.iloc[-1] / s.iloc[0]) ** (1.0 / n_years) - 1.0
    log_ret = np.log(s / s.shift(1)).dropna()
    vol_annual = log_ret.std() * np.sqrt(TPY)
    cum = s / s.iloc[0]
    dd = cum / cum.cummax() - 1.0
    yearly = s.resample("YE").last().pct_change().dropna()
    return {
        "CAGR": cagr, "Vol_anual": vol_annual, "Max_DD": dd.min(),
        "Sharpe_ish": cagr / vol_annual if vol_annual > 1e-12 else np.nan,
        "Mejor_año": yearly.max() if len(yearly) > 0 else 0.0,
        "Peor_año": yearly.min() if len(yearly) > 0 else 0.0,
    }


stats_rows = []
for asset in synth.columns:
    s = asset_stats(synth[asset])
    stats_rows.append({
        "Activo": asset,
        "Periodo": f"{synth.index[0].year}–{synth.index[-1].year}",
        "CAGR": f"{s['CAGR']*100:.2f}%",
        "Vol anualizada": f"{s['Vol_anual']*100:.2f}%",
        "Max Drawdown": f"{s['Max_DD']*100:.2f}%",
        "Sharpe-ish (CAGR/vol)": f"{s['Sharpe_ish']:.2f}" if not np.isnan(s["Sharpe_ish"]) else "—",
        "Mejor año": f"{s['Mejor_año']*100:.1f}%",
        "Peor año": f"{s['Peor_año']*100:.1f}%",
    })

# Anadir USDMXN sobre la serie completa (desde 2001) para el efecto largo plazo
usdmxn_full_series = usdmxn["USDMXN"].astype(float)
n_years_full = (usdmxn_full_series.index[-1] - usdmxn_full_series.index[0]).days / 365.25
cagr_full = (usdmxn_full_series.iloc[-1] / usdmxn_full_series.iloc[0]) ** (1.0 / n_years_full) - 1.0
logret_full = np.log(usdmxn_full_series / usdmxn_full_series.shift(1)).dropna()
vol_full = logret_full.std() * np.sqrt(TPY)
cum_full = usdmxn_full_series / usdmxn_full_series.iloc[0]
dd_full = cum_full / cum_full.cummax() - 1.0
yearly_full = usdmxn_full_series.resample("YE").last().pct_change().dropna()
stats_rows.insert(3, {
    "Activo": "USDT (en MXN) — serie larga",
    "Periodo": f"{usdmxn_full_series.index[0].year}–{usdmxn_full_series.index[-1].year}",
    "CAGR": f"{cagr_full*100:.2f}%",
    "Vol anualizada": f"{vol_full*100:.2f}%",
    "Max Drawdown": f"{dd_full.min()*100:.2f}%",
    "Sharpe-ish (CAGR/vol)": f"{cagr_full/vol_full:.2f}",
    "Mejor año": f"{yearly_full.max()*100:.1f}%",
    "Peor año": f"{yearly_full.min()*100:.1f}%",
})

historic_stats = pd.DataFrame(stats_rows).set_index("Activo")
print("ESTADISTICAS HISTORICAS")
print("=" * 80)
display(historic_stats)


In [ ]:
# --- Crecimiento acumulado ($1 invertido) ---
palette = {"USDT (en MXN)": "#003049", "BTC (en MXN)": "#f77f00",
           "ETH (en MXN)": "#d62828", "MXN cash (en MXN)": "#888",
           "USDT (USD)": "#4f9d69"}

fig, ax = plt.subplots(figsize=(13, 6))
for asset in ["USDT (en MXN)", "BTC (en MXN)", "ETH (en MXN)",
              "MXN cash (en MXN)", "USDT (USD)"]:
    s = synth[asset] / synth[asset].iloc[0]
    ax.plot(s.index, s.values, label=asset, color=palette[asset], linewidth=1.6)
ax.set_yscale("log")
ax.set_title("Crecimiento acumulado de $1 invertido (escala log)", fontweight="bold")
ax.set_ylabel("Múltiplo del capital inicial")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
fig.savefig(f"{IMG_DIR}/01_crecimiento_acumulado.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Drawdowns ---
fig, ax = plt.subplots(figsize=(13, 5))
for asset in ["USDT (en MXN)", "BTC (en MXN)", "ETH (en MXN)"]:
    s = synth[asset]
    cum = s / s.iloc[0]
    dd = cum / cum.cummax() - 1.0
    ax.fill_between(dd.index, dd.values * 100, 0, alpha=0.35,
                    color=palette[asset], label=asset)
ax.set_title("Drawdowns historicos (% caida desde peak)", fontweight="bold")
ax.set_ylabel("Drawdown (%)")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.0f}%"))
ax.legend(loc="lower left", fontsize=9)
plt.tight_layout()
fig.savefig(f"{IMG_DIR}/02_drawdowns.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Prediccion Monte Carlo del USD/MXN

Bootstrap-KDE sobre retornos diarios del USDMXN (Banxico, 2001–2026), 5,000 trayectorias.


In [ ]:
usdmxn_full = usdmxn["USDMXN"].astype(float)
fx_now = float(usdmxn_full.iloc[-1])
mc_horizons = {}

# Simular el horizonte mas largo y muestrear los demas para coherencia
horizon_max_years = max(CONFIG["MC_HORIZONS_YEARS"])
horizon_steps_max = int(horizon_max_years * TPY)

mc = MonteCarloPriceSimulator(usdmxn_full, n_sims=CONFIG["MC_N_SIMS"])
paths_max = mc.simulate_paths(horizon_steps_max)

print(f"Trayectorias simuladas: {paths_max.shape[1]}, longitud {paths_max.shape[0]} pasos\n")
for years in CONFIG["MC_HORIZONS_YEARS"]:
    step = int(years * TPY)
    terminal = paths_max.iloc[step].values
    p5, p50, p95 = (float(np.percentile(terminal, q)) for q in (5, 50, 95))
    rel = terminal / fx_now - 1.0
    mc_horizons[years] = {
        "usdmxn_p5": p5, "usdmxn_p50": p50, "usdmxn_p95": p95,
        "prob_dev_5": float((rel >= 0.05).mean()),
        "prob_dev_10": float((rel >= 0.10).mean()),
        "prob_dev_20": float((rel >= 0.20).mean()),
        "prob_apr_5": float((rel <= -0.05).mean()),
        "prob_apr_10": float((rel <= -0.10).mean()),
        "terminal_samples": terminal,
    }
    print(f"  {years}y | P5=${p5:.2f}  P50=${p50:.2f}  P95=${p95:.2f}  "
          f"P(dev>=10%)={mc_horizons[years]['prob_dev_10']*100:.1f}%")


In [ ]:
# Trayectorias
fig, ax = plt.subplots(figsize=(13, 6))
for i in range(min(150, paths_max.shape[1])):
    s = paths_max.iloc[:, i]
    color = "#006d77" if s.iloc[-1] >= fx_now else "#e29578"
    ax.plot(s.values, alpha=0.25, linewidth=0.7, color=color)
ax.axhline(y=fx_now, color="black", linestyle="--", alpha=0.7,
           label=f"USDMXN actual = ${fx_now:.2f}")
ax.set_title(f"Trayectorias simuladas USDMXN — {horizon_max_years} años", fontweight="bold")
ax.set_xlabel("Días futuros")
ax.set_ylabel("USDMXN")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x:.2f}"))
ax.legend()
plt.tight_layout()
fig.savefig(f"{IMG_DIR}/03_mc_trayectorias.png", dpi=150, bbox_inches="tight")
plt.show()

# Bandas de percentiles
fig, ax = plt.subplots(figsize=(13, 6))
desc = paths_max.T.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
xs = np.arange(len(desc.loc["50%"]))
ax.fill_between(xs, desc.loc["5%"].values, desc.loc["95%"].values,
                alpha=0.15, color="#003049", label="P5–P95")
ax.fill_between(xs, desc.loc["25%"].values, desc.loc["75%"].values,
                alpha=0.30, color="#003049", label="P25–P75")
ax.plot(xs, desc.loc["50%"].values, color="#d62828", linewidth=2.0, label="Mediana")
ax.axhline(y=fx_now, color="black", linestyle="--", alpha=0.6,
           label=f"USDMXN actual = ${fx_now:.2f}")
for years in CONFIG["MC_HORIZONS_YEARS"]:
    ax.axvline(x=years * TPY, color="gray", linestyle=":", alpha=0.5)
    ax.text(years * TPY, ax.get_ylim()[1] * 0.97, f"{years}y",
            ha="center", fontsize=9, color="gray")
ax.set_title("Banda de percentiles del USDMXN proyectado", fontweight="bold")
ax.set_xlabel("Días futuros")
ax.set_ylabel("USDMXN")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x:.2f}"))
ax.legend(loc="upper left")
plt.tight_layout()
fig.savefig(f"{IMG_DIR}/04_mc_percentiles.png", dpi=150, bbox_inches="tight")
plt.show()

# Distribucion terminal por horizonte
fig, axes = plt.subplots(1, len(CONFIG["MC_HORIZONS_YEARS"]),
                          figsize=(4.5 * len(CONFIG["MC_HORIZONS_YEARS"]), 4),
                          sharey=True)
for i, years in enumerate(CONFIG["MC_HORIZONS_YEARS"]):
    h = mc_horizons[years]
    ax = axes[i]
    ax.hist(h["terminal_samples"], bins=60, color="#003049", alpha=0.75, edgecolor="white")
    ax.axvline(x=fx_now, color="red", linestyle="--", alpha=0.8,
               label=f"Hoy = ${fx_now:.2f}")
    ax.axvline(x=h["usdmxn_p50"], color="black", linestyle="-", alpha=0.8,
               label=f"P50 = ${h['usdmxn_p50']:.2f}")
    ax.set_title(f"{years} año{'s' if years > 1 else ''}", fontweight="bold")
    ax.set_xlabel("USDMXN terminal")
    if i == 0:
        ax.set_ylabel("Frecuencia")
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x:.1f}"))
    ax.legend(fontsize=8)
fig.suptitle("Distribucion terminal del USDMXN por horizonte", fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig(f"{IMG_DIR}/05_mc_terminal.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Estrategias de yield sobre USDT

Cinco estrategias con yield USD anual tipico. Retorno MXN-equivalente combina yield + apreciacion del USDMXN bajo escenarios P5/P50/P95 (1 ano).


In [ ]:
h1 = mc_horizons[1]
factor_p5  = h1["usdmxn_p5"]  / fx_now
factor_p50 = h1["usdmxn_p50"] / fx_now
factor_p95 = h1["usdmxn_p95"] / fx_now

yield_rows = []
for label, apy, comment in CONFIG["YIELD_STRATEGIES"]:
    usd_grown = (1 + apy)
    yield_rows.append({
        "Estrategia": label,
        "Yield USD anual (APY)": f"{apy*100:.1f}%",
        "Retorno MXN P5 (1y)": f"{(usd_grown*factor_p5 - 1.0)*100:.1f}%",
        "Retorno MXN mediano (1y)": f"**{(usd_grown*factor_p50 - 1.0)*100:.1f}%**",
        "Retorno MXN P95 (1y)": f"{(usd_grown*factor_p95 - 1.0)*100:.1f}%",
        "Notas": comment,
    })

yield_scenarios = pd.DataFrame(yield_rows).set_index("Estrategia")
display(yield_scenarios)

# Grafica de comparacion (rango P5-P95 con mediana)
labels_short = [s[:35] + ("…" if len(s) > 35 else "") for s, _, _ in CONFIG["YIELD_STRATEGIES"]]
y_pos = np.arange(len(labels_short))
p5_vals  = [(((1+apy)*factor_p5 ) - 1.0)*100 for _, apy, _ in CONFIG["YIELD_STRATEGIES"]]
p50_vals = [(((1+apy)*factor_p50) - 1.0)*100 for _, apy, _ in CONFIG["YIELD_STRATEGIES"]]
p95_vals = [(((1+apy)*factor_p95) - 1.0)*100 for _, apy, _ in CONFIG["YIELD_STRATEGIES"]]

fig, ax = plt.subplots(figsize=(12, 6))
for i, (p5, p50, p95) in enumerate(zip(p5_vals, p50_vals, p95_vals)):
    ax.plot([p5, p95], [i, i], color="#003049", linewidth=8, alpha=0.5,
            solid_capstyle="round")
    ax.plot(p50, i, "o", color="#d62828", markersize=12,
            markeredgecolor="white", markeredgewidth=2, zorder=5)
ax.axvline(x=0, color="gray", linestyle="--", alpha=0.6)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels_short, fontsize=10)
ax.invert_yaxis()
ax.set_xlabel("Retorno MXN-equivalente a 1 año (%)")
ax.set_title("Retorno MXN-equivalente por estrategia de yield USDT (rango P5–P95)\n"
             "Punto rojo = mediana proyectada", fontweight="bold")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.0f}%"))
plt.tight_layout()
fig.savefig(f"{IMG_DIR}/06_yield_strategies.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Generacion del reporte

Genera `reporte_usdt_analysis.md` y un PDF (o HTML imprimible si weasyprint no esta disponible).


In [ ]:
from generate_report import write_report
from generate_pdf import md_to_pdf

fx_summary = {
    "start_date": str(usdmxn_full.index.min().date()),
    "end_date": str(usdmxn_full.index.max().date()),
    "start_fx": float(usdmxn_full.iloc[0]),
    "end_fx": fx_now,
    "total_appreciation": float(fx_now / usdmxn_full.iloc[0] - 1.0),
}
common_period = {"start": str(common_start.date()), "end": str(common_end.date())}
rec_idx = CONFIG["YIELD_RECOMMENDED_INDEX"]

write_report(
    historic_stats=historic_stats,
    mc_horizons=mc_horizons,
    yield_scenarios=yield_scenarios,
    fx_summary=fx_summary,
    common_period=common_period,
    capital_usd=CONFIG["CAPITAL_USD"],
    img_dir=IMG_DIR,
    yield_recommended_label=CONFIG["YIELD_STRATEGIES"][rec_idx][0],
    yield_recommended_apy=CONFIG["YIELD_STRATEGIES"][rec_idx][1],
)

md_to_pdf("reporte_usdt_analysis.md", "reporte_usdt_analysis.pdf")
